# Figures as interfaces: NYC taxi exploration

For BADM 554 students also taking BDI 513 Data Storytelling. We use a real, deterministic 50,000-trip sample from January 2024. Counts are sample counts, not full-month totals. The first figure shows Manhattan pickups by hour. An evening brush ranks pickup zones; a morning brush updates the same linked chart.

## Setup in Google Colab

Obtain this repository revision as a ZIP from the instructor, including `data/`, or clone a published revision into `/content/figures-as-interfaces`. The first cell uses that folder when present; otherwise it asks you to upload the repository ZIP, then installs the package from it. Package installation needs internet; all analysis afterward is offline and needs no API key. Locally, run from the repository or `examples/` directory.


In [ ]:
from pathlib import Path
import sys, subprocess

REPO = Path('/content/figures-as-interfaces')
if not REPO.exists():
    local = Path.cwd()
    if (local / 'pyproject.toml').exists():
        REPO = local
    elif (local.parent / 'pyproject.toml').exists():
        REPO = local.parent
    else:
        from google.colab import files
        import io, zipfile
        uploaded = files.upload()
        archive_name = next(iter(uploaded))
        with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
            archive.extractall('/content/figureflow-upload')
        candidates = list(Path('/content/figureflow-upload').rglob('pyproject.toml'))
        if len(candidates) != 1:
            raise ValueError('Upload the FigureFlow repository ZIP with one pyproject.toml')
        REPO = candidates[0].parent
subprocess.run([sys.executable, '-m', 'pip', 'install', str(REPO)], check=True)


## Run the linked exploration

This command creates the hourly chart, selects hours 17 through 20 inclusive, asks for a zone ranking, then selects hours 7 through 10 inclusive. The second brush uses the stored coordination rule, with no new planner call. Old and new rankings both remain in the artifact.


In [ ]:
import os, json, subprocess, sys
from pathlib import Path

OUTPUT = Path(os.environ.get('FIGUREFLOW_DEMO_OUTPUT', str(REPO / 'demo-output')))
subprocess.run([sys.executable, '-m', 'figureflow', 'demo', '--output', str(OUTPUT)], check=True)
artifact = json.loads((OUTPUT / 'artifact.json').read_text())
print('Artifact versions:', list(artifact['versions']))


## Render saved specifications

Read the language-neutral JSON and render each Vega-Lite specification. This cell imports no FigureFlow code. Wolfram can use the same specs with a compatible Vega-Lite renderer.


In [ ]:
from IPython.display import Image, display
import vl_convert as vlc

for figure in artifact['figures'].values():
    print(figure['M']['operation'], figure['V']['summary'][:180])
    display(Image(vlc.vegalite_to_png(figure['V']['spec'], vl_version='5.20', allowed_base_urls=[])))


## Map selected marks back to rows without importing the package

A future visual brush adapter supplies these `mark_id` values. Here we select them directly from the displayed data. Every contributing trip is recoverable through the explicit map, including marks that aggregate many trips. The normalized rows include original Parquet row ids; the committed sample retains the full raw trip columns.


In [ ]:
first_state = artifact['versions']['a0001']
first_figure = artifact['figures'][first_state['figures']['chart0001']]
selected_marks = [row['mark_id'] for row in first_figure['V']['spec']['data']['values']
                  if 17 <= row['pickup_hour'] <= 20]
row_ids = {row_id for mark in selected_marks for row_id in first_figure['R']['mark_to_rows'][mark]}
selected_rows = [row for row in first_figure['D']['rows'] if row['row_id'] in row_ids]
print('Selected trips:', len(selected_rows))
print('Selected hours:', sorted({row['pickup_hour'] for row in selected_rows}))
print('Example trip:', selected_rows[0])


## Verify saved history

Re-execute all stored SQL against the embedded input snapshot and compare the resulting rows, counts, mark mappings and specifications. No original course folder or full Parquet file is needed.


In [ ]:
subprocess.run([sys.executable, '-m', 'figureflow', 'replay', str(OUTPUT / 'artifact.json')], check=True)
